# Explainability Demo — Strategy Saliency & Feature Importance

This notebook demonstrates all three explainability techniques from **Epic E5.3**:

1. **Gradient saliency** — which observation dimensions most affect the policy output.
2. **Integrated gradients** — attribution scores that sum to the policy output difference.
3. **SHAP / permutation importance** — model-agnostic feature ranking.

The notebook runs end-to-end from a saved checkpoint **or** in demo mode with a
randomly-initialised policy when no checkpoint is available.

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to sys.path so local imports work from notebooks/ dir
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib
matplotlib.use("Agg")  # use non-interactive backend in headless environments
import matplotlib.pyplot as plt

from analysis.saliency import (
    OBSERVATION_FEATURES,
    SaliencyAnalyzer,
    plot_saliency_map,
    plot_feature_importance,
)
from envs.battalion_env import BattalionEnv

print(f'Project root : {PROJECT_ROOT}')
print(f'Obs features : {OBSERVATION_FEATURES}')

## Configuration

Set `CHECKPOINT_PATH` to a saved SB3 `.zip` file to use a trained policy.  
Leave as `None` to run in **demo mode** with a randomly-initialised policy.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Path to an SB3 PPO checkpoint, or None for demo mode.
CHECKPOINT_PATH = None   # e.g. Path('../checkpoints/my_run/final')

MAX_STEPS     = 200    # episode length
N_TRAJ_STEPS  = 50     # trajectory steps to analyse
SEED          = 42

## Load Policy

In [ ]:
if CHECKPOINT_PATH is not None:
    from stable_baselines3 import PPO
    policy = PPO.load(str(CHECKPOINT_PATH))
    print(f'Loaded policy from {CHECKPOINT_PATH}')
else:
    from stable_baselines3 import PPO
    from models.mlp_policy import BattalionMlpPolicy
    _env = BattalionEnv(randomize_terrain=False, max_steps=MAX_STEPS)
    policy = PPO(BattalionMlpPolicy, _env, verbose=0)
    print('No checkpoint — using randomly-initialised policy (demo mode).')

## Collect a Trajectory

Run the policy for `N_TRAJ_STEPS` steps to collect observations for analysis.

In [ ]:
env = BattalionEnv(randomize_terrain=False, max_steps=MAX_STEPS)
obs, _ = env.reset(seed=SEED)

trajectory = [obs.copy()]
for _ in range(N_TRAJ_STEPS - 1):
    action, _ = policy.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = env.step(action)
    trajectory.append(obs.copy())
    if terminated or truncated:
        obs, _ = env.reset()

traj_obs = np.stack(trajectory)  # (N_TRAJ_STEPS, 12)
print(f'Collected trajectory: shape={traj_obs.shape}')
print(f'\nObservation statistics over {N_TRAJ_STEPS} steps:')
for i, name in enumerate(OBSERVATION_FEATURES):
    col = traj_obs[:, i]
    print(f'  [{i:2d}] {name:<16}  mean={col.mean():.3f}  std={col.std():.3f}')

## 1. Gradient Saliency

Gradient saliency back-propagates through the policy and uses the absolute input gradient as an importance proxy.

In [ ]:
analyzer = SaliencyAnalyzer(policy)

saliency = analyzer.gradient_saliency(traj_obs)

print('Gradient saliency scores:')
for name, score in zip(OBSERVATION_FEATURES, saliency):
    bar = '#' * int(score / max(saliency) * 30)
    print(f'  {name:<16} {score:.4f}  {bar}')

print('\nTop-3 most salient features:')
for rank, (name, score) in enumerate(analyzer.top_features(saliency, k=3), 1):
    print(f'  {rank}. {name}  ({score:.4f})')

In [ ]:
fig = analyzer.plot_saliency(saliency, title='Gradient Saliency — Trajectory Mean')
plt.show()
plt.close('all')

## 2. Integrated Gradients

Integrated gradients accumulate gradients along the path from a zero baseline to the observation, giving attributions that satisfy the *completeness* axiom.

In [ ]:
ig_scores = analyzer.integrated_gradients(traj_obs, n_steps=50)

print('Integrated gradient attributions (trajectory mean):')
for name, score in zip(OBSERVATION_FEATURES, ig_scores):
    print(f'  {name:<16}  {score:+.4f}')

print('\nTop-3 by absolute attribution:')
for rank, (name, score) in enumerate(analyzer.top_features(np.abs(ig_scores), k=3), 1):
    print(f'  {rank}. {name}  ({score:.4f})')

In [ ]:
fig = analyzer.plot_saliency(
    np.abs(ig_scores),
    title='Integrated Gradients — |Attribution| (Trajectory Mean)',
)
plt.show()
plt.close('all')

## 3. SHAP / Permutation Feature Importance

When the `shap` package is installed this uses `GradientExplainer`. Otherwise it falls back to a permutation-based approximation.

In [ ]:
shap_scores = analyzer.shap_importance(traj_obs)

print('SHAP / permutation importance scores:')
for name, score in zip(OBSERVATION_FEATURES, shap_scores):
    print(f'  {name:<16}  {score:.4f}')

print('\nTop-3 most important features:')
for rank, (name, score) in enumerate(analyzer.top_features(shap_scores, k=3), 1):
    print(f'  {rank}. {name}  ({score:.4f})')

In [ ]:
fig = analyzer.plot_importance(shap_scores, title='SHAP / Permutation Importance')
plt.show()
plt.close('all')

## 4. Combined Summary Plot

All three importance estimates side-by-side.

In [ ]:
summary = analyzer.summary(traj_obs, n_steps=30, n_samples=20)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'darkorange', 'seagreen']
method_titles = [
    ('gradient_saliency',    'Gradient Saliency'),
    ('integrated_gradients', 'Integrated Gradients'),
    ('shap_importance',      'SHAP / Permutation'),
]

for ax, (key, title), color in zip(axes, method_titles, colors):
    scores = np.abs(summary[key])
    norm_scores = scores / (scores.max() + 1e-9)
    y = np.arange(len(OBSERVATION_FEATURES))
    ax.barh(y, norm_scores, color=color, edgecolor='white')
    ax.set_yticks(y)
    ax.set_yticklabels(OBSERVATION_FEATURES, fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Normalised importance', fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.1)

fig.suptitle('Strategy Explainability — All Methods', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()
plt.close('all')

## 5. Observation Reference

For a full description of every observation dimension see
[`docs/observation_reference.md`](../docs/observation_reference.md).